In [1]:
# Celda 1 — Verificar que el entorno funciona correctamente
# Importamos las librerías principales del proyecto

import pandas as pd               # manejo de datos en tablas (DataFrames)
import matplotlib.pyplot as plt   # creación de gráficas
from pydataxm.pydataxm import ReadDB  # conexión con la API de XM

# Si esta celda corre sin errores, el entorno está configurado correctamente
print("Entorno configurado correctamente")
print(f"Versión de pandas: {pd.__version__}")

Entorno configurado correctamente
Versión de pandas: 3.0.2


In [2]:
# Celda — Parche de compatibilidad entre pydataxm 0.3.17 y pandas 3.0
#
# pydataxm fue escrita para una version de pandas anterior a varias limpiezas de API
# que llegaron con pandas 3.0. Dentro de request_data() encontramos TRES puntos
# de choque, todos del mismo tipo (parametros/alias que pandas elimino):
#   1) pd.date_range(..., freq='M', ...)          -> 'M' ya no existe, ahora es 'ME'
#   2) pd.to_numeric(..., errors='ignore')        -> 'ignore' ya no es valor valido
#   3) pd.to_datetime(..., errors='ignore', ...)  -> mismo problema que el anterior
#
# Parcheamos las tres funciones SOLO en esta sesion de Python (este notebook),
# sin tocar el archivo de la libreria instalada. Si pydataxm se actualiza en el
# futuro para soportar pandas 3.0, este parche se puede borrar sin dejar rastro.

import pandas as pd

# --- Parche 1: pd.date_range con freq='M' ---
_date_range_original = pd.date_range

def _date_range_parchado(*args, **kwargs):
    if kwargs.get("freq") == "M":
        kwargs["freq"] = "ME"
    return _date_range_original(*args, **kwargs)

pd.date_range = _date_range_parchado

# --- Parche 2: pd.to_numeric con errors='ignore' ---
_to_numeric_original = pd.to_numeric

def _to_numeric_parchado(*args, **kwargs):
    if kwargs.get("errors") == "ignore":
        kwargs["errors"] = "coerce"
    return _to_numeric_original(*args, **kwargs)

pd.to_numeric = _to_numeric_parchado

# --- Parche 3: pd.to_datetime con errors='ignore' ---
_to_datetime_original = pd.to_datetime

def _to_datetime_parchado(*args, **kwargs):
    if kwargs.get("errors") == "ignore":
        # 'coerce' convierte lo que pueda a fecha real; lo que no pueda
        # lo deja como NaT (Not a Time, el "NaN" de las fechas en pandas).
        kwargs["errors"] = "coerce"
    return _to_datetime_original(*args, **kwargs)

pd.to_datetime = _to_datetime_parchado

print("Parches aplicados: date_range (M->ME), to_numeric e to_datetime (ignore->coerce)")

Parches aplicados: date_range (M->ME), to_numeric e to_datetime (ignore->coerce)


In [3]:
# Celda 2 — Primera conexión con la API de XM
# ReadDB es la clase principal de pydataxm para conectarse con los servidores de XM
api = ReadDB()

# Descargamos el precio de bolsa de la primera semana de enero 2024
# Parámetros:
#   - "PrecioEscasez": nombre del indicador que queremos (precio de bolsa horario)
#   - "Sistema": nivel de agregación nacional
#   - fecha inicio y fecha fin en formato "YYYY-MM-DD"
precio = api.request_data(
    "PrecioEscasez",
    "Sistema",
    "2024-01-01",
    "2024-01-07"
)

# Mostramos las primeras filas para ver cómo llegaron los datos
print(type(precio))   # confirma que es un DataFrame
print(precio.shape)   # muestra cuántas filas y columnas tiene
print(precio.head())  # muestra las primeras 5 filas

No existe la métrica PrecioEscasez
<class 'pandas.DataFrame'>
(0, 0)
Empty DataFrame
Columns: []
Index: []


In [4]:
# Celda 3 — Ver todos los indicadores disponibles en la API de XM
# Esto nos muestra el catálogo completo de métricas que podemos consultar
catalogo = api.get_collections()
print(catalogo)

            MetricId                                         MetricName  \
0           DemaReal                           Demanda Real por Sistema   
1           DemaReal                            Demanda Real por Agente   
2         ExpoMoneda                   Exportaciones Moneda por Sistema   
3           DemaCome                      Demanda Comercial por Sistema   
4               Gene                             Generación por Sistema   
..               ...                                                ...   
188   ListadoAgentes       Listado de agentes con atributos por Sistema   
189      ListadoRios                        Listado de rios por Sistema   
190  ListadoEmbalses                    Listado de embalses por Sistema   
191  ListadoMetricas                    Listado de Métricas por Sistema   
192      ListadoAGPE  Listado Autogeneradores a Pequeña Escala por A...   

      Entity  MaxDays            Type                                 Url  \
0    Sistema       31 

In [5]:
import sys
print(sys.executable)

c:\Users\mgdbj\xm-spot-price-predictor\venv\Scripts\python.exe


In [6]:
# ReadDB es la clase principal de la libreria pydataxm.
from pydataxm.pydataxm import ReadDB

# pandas para manejar el catalogo como tabla ('pd' es su alias estandar).
import pandas as pd

# Creamos el objeto que se conecta a la API de XM (el constructor no recibe argumentos).
objeto_api = ReadDB()

# Traemos el catalogo completo de metricas. get_collections() es metodo de pydataxm
# y devuelve un DataFrame de pandas.
catalogo = objeto_api.get_collections()

# Primero vemos como se llaman las columnas del catalogo, para saber donde buscar.
print(catalogo.columns.tolist())

# Buscamos en TODAS las columnas de texto cualquier fila que mencione "Bolsa Nacional".
# .apply recorre columnas; .astype(str).str.contains es de pandas (busca el texto).
mascara = catalogo.apply(lambda columna: columna.astype(str).str.contains("Bolsa Nacional", case=False, na=False))

# Mostramos solo las filas que coinciden en alguna columna.
print(catalogo[mascara.any(axis=1)])

['MetricId', 'MetricName', 'Entity', 'MaxDays', 'Type', 'Url', 'Filter', 'MetricUnits', 'MetricDescription']
                MetricId                                        MetricName  \
6           PrecBolsNaci                 Precio Bolsa Nacional por Sistema   
15      CompBolsNaciEner        Compras Bolsa Nacional Energía por Sistema   
16      CompBolsNaciEner         Compras Bolsa Nacional Energía por Agente   
67       PrecBolsNaciTX1  Precio Bolsa Promedio Aritmético TX1 por Sistema   
88    CompBolsaNacMoneda         Compras Bolsa Nacional Moneda por Sistema   
90   VentaBolsNaciMoneda          Ventas Bolsa Nacional Moneda por Sistema   
91      VentBolsNaciEner         Ventas Bolsa Nacional Energía por Sistema   
92      VentBolsNaciEner          Ventas Bolsa Nacional Energía por Agente   
116             DeltaNal             Delta Incremento Nacional por Sistema   
166       PPPrecBolsNaci       Precio Bolsa Nacional Ponderado por Sistema   

      Entity  MaxDays           

In [7]:
# datetime es libreria estandar de Python (no se instala). 'dt' es su alias comun.
import datetime as dt

# MetricId y Entity tomados del catalogo de la celda anterior.
metric_id = "PrecBolsNal"   # <-- cambialo si el catalogo mostro otro codigo
entity = "Sistema"          # cruce a nivel de todo el sistema

# Rango de PRUEBA: solo 3 dias (recuerda: maximo 30 dias por llamado en datos horarios).
fecha_inicio = dt.date(2024, 1, 1)   # objeto date de datetime
fecha_fin = dt.date(2024, 1, 3)

# request_data devuelve un DataFrame de pandas.
# Ojo: por orden, el 1er argumento es el MetricId y el 2do es el Entity
# (aunque internamente los parametros tengan otros nombres).
df_precio = objeto_api.request_data(metric_id, entity, fecha_inicio, fecha_fin)

# Vemos el tamaño (filas, columnas) y las primeras filas para inspeccionar.
print(df_precio.shape)
df_precio.head()

No existe la métrica PrecBolsNal
(0, 0)


""


In [8]:
# Celda 4 — Descarga de prueba: Precio de Bolsa Nacional (3 dias, para validar)

# datetime es libreria estandar de Python (no se instala). 'dt' es su alias comun.
import datetime as dt

# MetricId y Entity confirmados en el catalogo de la celda anterior.
metric_id = "PrecBolsNaci"
entity = "Sistema"

# Rango de PRUEBA: solo 3 dias (recuerda: maximo 31 dias por llamado para datos horarios).
fecha_inicio = dt.date(2024, 1, 1)
fecha_fin = dt.date(2024, 1, 3)

# request_data es metodo de pydataxm (clase ReadDB) y devuelve un DataFrame de pandas.
df_precio = api.request_data(metric_id, entity, fecha_inicio, fecha_fin)

# Vemos el tamaño (filas, columnas) y las primeras filas para inspeccionar la forma del dato.
print(df_precio.shape)
df_precio.head()

(3, 27)


,Id,Values_code,Values_Hour01,Values_Hour02,Values_Hour03,Values_Hour04,Values_Hour05,Values_Hour06,Values_Hour07,Values_Hour08,...,Values_Hour16,Values_Hour17,Values_Hour18,Values_Hour19,Values_Hour20,Values_Hour21,Values_Hour22,Values_Hour23,Values_Hour24,Date
0,NaN,NaN,251.91161,251.91161,251.91161,281.91161,281.91161,251.91161,152.91161,95.85161,...,251.91161,281.91161,281.91161,301.23261,301.23261,301.23261,301.23261,301.23261,290.91161,NaT
1,NaN,NaN,360.22127,360.22127,309.82127,284.32127,309.82127,317.32127,317.32127,360.22127,...,419.32127,419.32127,419.32127,449.32127,449.32127,449.32127,449.32127,419.32127,419.32127,NaT
2,NaN,NaN,429.48823,429.48823,429.48823,429.48823,429.48823,429.48823,429.48823,429.48823,...,504.48823,504.48823,509.48823,509.48823,514.48823,509.48823,509.48823,504.48823,504.48823,NaT


In [9]:
# Celda — Diagnostico: que trae REALMENTE la columna Date antes de la conversion
#
# Vamos a llamar a la API "a mano", sin pasar por el metodo request_data(),
# para ver el dato crudo tal como lo entrega el servidor, sin que ningun
# parche ni ninguna conversion lo toque todavia.

import requests  # libreria estandar para hacer peticiones HTTP, ya viene con pydataxm como dependencia
import json

# Body de la peticion: mismo formato que vimos en la documentacion oficial de XM.
cuerpo_peticion = {
    "MetricId": "PrecBolsNaci",
    "StartDate": "2024-01-01",
    "EndDate": "2024-01-03",
    "Entity": "Sistema"
}

# Hacemos la peticion directamente al endpoint horario (sin pasar por pydataxm).
respuesta = requests.post("https://servapibi.xm.com.co/hourly", json=cuerpo_peticion)

# Convertimos la respuesta a un diccionario de Python para inspeccionarla.
datos_crudos = respuesta.json()

# Imprimimos solo el primer registro, para ver como viene la fecha de verdad.
print(json.dumps(datos_crudos["Items"][0], indent=2, ensure_ascii=False))

{
  "Date": "2024-01-01",
  "HourlyEntities": [
    {
      "Id": "Sistema",
      "Values": {
        "code": "Sistema",
        "Hour01": "251.91161",
        "Hour02": "251.91161",
        "Hour03": "251.91161",
        "Hour04": "281.91161",
        "Hour05": "281.91161",
        "Hour06": "251.91161",
        "Hour07": "152.91161",
        "Hour08": "95.85161",
        "Hour09": "95.85161",
        "Hour10": "95.85161",
        "Hour11": "95.85161",
        "Hour12": "152.91161",
        "Hour13": "152.91161",
        "Hour14": "152.91161",
        "Hour15": "152.91161",
        "Hour16": "251.91161",
        "Hour17": "281.91161",
        "Hour18": "281.91161",
        "Hour19": "301.23261",
        "Hour20": "301.23261",
        "Hour21": "301.23261",
        "Hour22": "301.23261",
        "Hour23": "301.23261",
        "Hour24": "290.91161"
      }
    }
  ]
}


In [10]:
# Celda — Funcion propia para descargar y aplanar el precio de bolsa
#
# Por que existe esta funcion: el metodo request_data() de pydataxm 0.3.17 tiene
# un bug al aplanar la estructura anidada del JSON de XM (la columna Date no
# queda alineada con las filas de precio). En vez de seguir parchando su logica
# interna, construimos nuestra propia funcion, usando requests directamente
# contra el endpoint oficial. Esto nos da control total sobre el resultado.

import requests   # libreria estandar para hacer peticiones HTTP
import pandas as pd

def descargar_precio_bolsa(fecha_inicio, fecha_fin):
    """
    Descarga el Precio de Bolsa Nacional (COP/kWh) para un rango de fechas.
    Maximo 31 dias por llamado (restriccion de la API de XM).

    Parametros:
        fecha_inicio, fecha_fin: strings en formato 'YYYY-MM-DD'

    Devuelve:
        DataFrame de pandas en formato LARGO: una fila por cada hora,
        con columnas 'fecha_hora' y 'precio_bolsa_cop_kwh'.
    """
    # Cuerpo de la peticion, igual al que vimos en la documentacion oficial de XM.
    cuerpo_peticion = {
        "MetricId": "PrecBolsNaci",
        "StartDate": fecha_inicio,
        "EndDate": fecha_fin,
        "Entity": "Sistema"
    }

    # Hacemos la peticion POST al endpoint horario.
    respuesta = requests.post("https://servapibi.xm.com.co/hourly", json=cuerpo_peticion)
    respuesta.raise_for_status()  # lanza un error claro si la peticion HTTP fallo

    datos = respuesta.json()

    # Aqui hacemos el aplanado NOSOTROS MISMOS, con control total sobre el resultado.
    filas = []  # lista donde vamos a acumular un diccionario por cada hora real

    # 'Items' contiene un objeto por cada dia del rango solicitado.
    for dia in datos["Items"]:
        fecha = dia["Date"]  # ej. '2024-01-01', viene en el nivel superior de cada dia

        # 'HourlyEntities' es una lista; para Entity='Sistema' trae un solo elemento.
        valores_del_dia = dia["HourlyEntities"][0]["Values"]

        # Recorremos las 24 horas y construimos una fila por cada una.
        for hora in range(1, 25):
            clave_hora = f"Hour{hora:02d}"  # ej. 'Hour01', 'Hour02', ... 'Hour24'
            precio = float(valores_del_dia[clave_hora])  # viene como texto, lo pasamos a numero

            # Construimos la marca de tiempo exacta combinando fecha + hora.
            # hora-1 porque Hour01 representa el periodo que empieza a las 00:00.
            fecha_hora = pd.Timestamp(fecha) + pd.Timedelta(hours=hora - 1)

            filas.append({"fecha_hora": fecha_hora, "precio_bolsa_cop_kwh": precio})

    # Convertimos la lista de diccionarios en un DataFrame de pandas.
    return pd.DataFrame(filas)


# --- Prueba con el mismo rango de antes, para comparar ---
df_precio = descargar_precio_bolsa("2024-01-01", "2024-01-03")

print(df_precio.shape)
df_precio.head(10)

(72, 2)


,fecha_hora,precio_bolsa_cop_kwh
0,2024-01-01 00:00:00,251.91161
1,2024-01-01 01:00:00,251.91161
2,2024-01-01 02:00:00,251.91161
3,2024-01-01 03:00:00,281.91161
4,2024-01-01 04:00:00,281.91161
5,2024-01-01 05:00:00,251.91161
6,2024-01-01 06:00:00,152.91161
7,2024-01-01 07:00:00,95.85161
8,2024-01-01 08:00:00,95.85161
9,2024-01-01 09:00:00,95.85161


In [11]:
# Celda — Descarga por bloques (PRUEBA: 3 meses, enero-marzo 2024)
#
# La API de XM limita cada llamado a maximo 31 dias (Type=HourlyEntities).
# Para pedir un rango mas largo, lo partimos en bloques de 30 dias (dejamos
# 1 dia de margen de seguridad respecto al limite real de 31) y vamos
# acumulando los resultados de cada bloque en una lista.

import pandas as pd
import time  # libreria estandar de Python, la usamos para pausar entre llamadas

def descargar_rango_completo(fecha_inicio_total, fecha_fin_total, dias_por_bloque=30):
    """
    Descarga el precio de bolsa para un rango largo, dividiendolo en bloques
    que respetan el limite de la API (maximo 31 dias por llamado).

    Parametros:
        fecha_inicio_total, fecha_fin_total: strings 'YYYY-MM-DD'
        dias_por_bloque: tamaño de cada bloque (30 por defecto, con margen de seguridad)

    Devuelve:
        Un solo DataFrame con todo el rango, ya unido y ordenado por fecha_hora.
    """
    # Convertimos los strings a objetos Timestamp de pandas, para poder sumarles dias.
    fecha_actual = pd.Timestamp(fecha_inicio_total)
    fecha_limite = pd.Timestamp(fecha_fin_total)

    bloques_descargados = []  # aqui acumulamos un DataFrame por cada bloque exitoso
    numero_bloque = 1

    # El bucle se repite mientras la fecha de inicio del bloque no supere la fecha final total.
    while fecha_actual <= fecha_limite:
        # El fin de este bloque es 'dias_por_bloque - 1' dias despues del inicio,
        # pero sin pasarse de la fecha limite total.
        fin_de_bloque = min(fecha_actual + pd.Timedelta(days=dias_por_bloque - 1), fecha_limite)

        inicio_str = fecha_actual.strftime("%Y-%m-%d")
        fin_str = fin_de_bloque.strftime("%Y-%m-%d")

        print(f"Bloque {numero_bloque}: descargando {inicio_str} a {fin_str} ...")

        # Reutilizamos la funcion que ya construimos y validamos en el paso anterior.
        df_bloque = descargar_precio_bolsa(inicio_str, fin_str)
        bloques_descargados.append(df_bloque)

        # Avanzamos la fecha actual al dia siguiente del bloque que acabamos de pedir.
        fecha_actual = fin_de_bloque + pd.Timedelta(days=1)
        numero_bloque += 1

        # Pausa breve entre llamadas, para no saturar el servidor de XM con
        # peticiones demasiado seguidas (buena practica al consumir APIs publicas).
        time.sleep(1)

    # pd.concat junta todos los DataFrames de la lista en uno solo, uno debajo del otro.
    # ignore_index=True renumera las filas de 0 en adelante (si no, se repetirian indices).
    df_completo = pd.concat(bloques_descargados, ignore_index=True)

    return df_completo


# --- Prueba: enero a marzo de 2024 (3 meses) ---
df_prueba_3meses = descargar_rango_completo("2024-01-01", "2024-03-31")

print("\nResultado final:")
print(df_prueba_3meses.shape)
df_prueba_3meses.head()

Bloque 1: descargando 2024-01-01 a 2024-01-30 ...


ReadTimeout: HTTPSConnectionPool(host='servapibi.xm.com.co', port=443): Read timed out. (read timeout=None)

In [ ]:
# Celda — Verificaciones de integridad antes de guardar

# 1) ¿Hay fechas duplicadas? (podria pasar si dos bloques se solaparon por error)
duplicados = df_prueba_3meses["fecha_hora"].duplicated().sum()
print("Filas con fecha_hora duplicada:", duplicados)

# 2) ¿Hay huecos en la secuencia horaria? Generamos el rango horario "ideal"
# y lo comparamos contra las fechas que realmente descargamos.
rango_esperado = pd.date_range("2024-01-01 00:00", "2024-03-31 23:00", freq="h")
faltantes = rango_esperado.difference(df_prueba_3meses["fecha_hora"])
print("Horas faltantes:", len(faltantes))

Filas con fecha_hora duplicada: 0
Horas faltantes: 0


In [ ]:
# Celda — Version robusta de la descarga por bloques
#
# Mejoras respecto a la version anterior:
#   1) try/except con reintentos: si un bloque falla (servidor caido, internet,
#      etc.), no se pierde todo el avance; se reintenta ese bloque hasta 3 veces.
#   2) Devuelve tambien la lista de bloques que fallaron definitivamente, para
#      saber si quedo algun hueco que haya que volver a pedir despues.

import pandas as pd
import time  # libreria estandar, para las pausas entre llamadas

def descargar_rango_robusto(fecha_inicio_total, fecha_fin_total, dias_por_bloque=30, max_reintentos=3):
    """
    Descarga el precio de bolsa para un rango largo, en bloques, con reintentos.

    Devuelve una tupla de dos elementos:
        - df_completo: DataFrame con todo lo que se pudo descargar
        - bloques_fallidos: lista de (inicio, fin) de los bloques que fallaron del todo
    """
    fecha_actual = pd.Timestamp(fecha_inicio_total)
    fecha_limite = pd.Timestamp(fecha_fin_total)

    bloques_descargados = []   # DataFrames de los bloques exitosos
    bloques_fallidos = []      # rangos que no se pudieron descargar ni reintentando
    numero_bloque = 1

    while fecha_actual <= fecha_limite:
        fin_de_bloque = min(fecha_actual + pd.Timedelta(days=dias_por_bloque - 1), fecha_limite)
        inicio_str = fecha_actual.strftime("%Y-%m-%d")
        fin_str = fin_de_bloque.strftime("%Y-%m-%d")

        # Intentamos descargar este bloque, reintentando si falla.
        exito = False
        for intento in range(1, max_reintentos + 1):
            try:
                df_bloque = descargar_precio_bolsa(inicio_str, fin_str)
                bloques_descargados.append(df_bloque)
                print(f"Bloque {numero_bloque} OK: {inicio_str} a {fin_str} ({len(df_bloque)} filas)")
                exito = True
                break  # salimos del bucle de reintentos: ya funciono
            except Exception as error:
                # Si fallo, avisamos y esperamos antes de reintentar (espera creciente).
                print(f"  Bloque {numero_bloque} intento {intento} fallo: {error}")
                time.sleep(intento * 3)  # espera 3s, luego 6s, luego 9s

        # Si despues de todos los reintentos no lo logramos, lo anotamos como fallido.
        if not exito:
            bloques_fallidos.append((inicio_str, fin_str))
            print(f"  Bloque {numero_bloque} DESCARTADO tras {max_reintentos} intentos")

        fecha_actual = fin_de_bloque + pd.Timedelta(days=1)
        numero_bloque += 1
        time.sleep(1)  # pausa normal entre bloques

    # Unimos todo lo descargado (si hubo al menos un bloque exitoso).
    if bloques_descargados:
        df_completo = pd.concat(bloques_descargados, ignore_index=True)
        # Ordenamos por fecha y eliminamos posibles duplicados por seguridad.
        df_completo = df_completo.sort_values("fecha_hora").drop_duplicates("fecha_hora").reset_index(drop=True)
    else:
        df_completo = pd.DataFrame()

    return df_completo, bloques_fallidos

In [ ]:
# Celda — Mini-prueba: confirmar que existe dato en enero de 2019
df_test_2019 = descargar_precio_bolsa("2019-01-01", "2019-01-05")
print(df_test_2019.shape)
df_test_2019.head()

(120, 2)


,fecha_hora,precio_bolsa_cop_kwh
0,2019-01-01 00:00:00,318.4231
1,2019-01-01 01:00:00,318.4231
2,2019-01-01 02:00:00,318.4231
3,2019-01-01 03:00:00,318.4231
4,2019-01-01 04:00:00,308.9231


In [ ]:
# Celda — Descarga COMPLETA 2019-2024 y guardado en disco
#
# Esto hace ~73 llamadas a la API y puede tardar varios minutos.
# Veras el progreso bloque por bloque. Al final guarda el resultado en data/.

import pandas as pd

# Lanzamos la descarga robusta para el rango completo del proyecto.
df_precio_completo, fallidos = descargar_rango_robusto("2019-01-01", "2024-12-31")

# Reporte de resultado.
print("\n===== RESUMEN =====")
print("Total de filas descargadas:", df_precio_completo.shape[0])
print("Rango de fechas:", df_precio_completo["fecha_hora"].min(), "a", df_precio_completo["fecha_hora"].max())
print("Bloques que fallaron:", len(fallidos))
if fallidos:
    print("Rangos fallidos (habria que reintentarlos):", fallidos)

# Guardamos en la carpeta data/. to_csv es metodo de pandas que escribe a disco.
# index=False evita guardar el numero de fila como una columna extra innecesaria.
ruta_archivo = "../data/precio_bolsa_2019_2024.csv"
df_precio_completo.to_csv(ruta_archivo, index=False)
print(f"\nGuardado en: {ruta_archivo}")

Bloque 1 OK: 2019-01-01 a 2019-01-30 (720 filas)
Bloque 2 OK: 2019-01-31 a 2019-03-01 (720 filas)
Bloque 3 OK: 2019-03-02 a 2019-03-31 (720 filas)
Bloque 4 OK: 2019-04-01 a 2019-04-30 (720 filas)
Bloque 5 OK: 2019-05-01 a 2019-05-30 (720 filas)
Bloque 6 OK: 2019-05-31 a 2019-06-29 (720 filas)
Bloque 7 OK: 2019-06-30 a 2019-07-29 (720 filas)
Bloque 8 OK: 2019-07-30 a 2019-08-28 (720 filas)
Bloque 9 OK: 2019-08-29 a 2019-09-27 (720 filas)
Bloque 10 OK: 2019-09-28 a 2019-10-27 (720 filas)
Bloque 11 OK: 2019-10-28 a 2019-11-26 (720 filas)
Bloque 12 OK: 2019-11-27 a 2019-12-26 (720 filas)
Bloque 13 OK: 2019-12-27 a 2020-01-25 (720 filas)
Bloque 14 OK: 2020-01-26 a 2020-02-24 (720 filas)
Bloque 15 OK: 2020-02-25 a 2020-03-25 (720 filas)
Bloque 16 OK: 2020-03-26 a 2020-04-24 (720 filas)
Bloque 17 OK: 2020-04-25 a 2020-05-24 (720 filas)
Bloque 18 OK: 2020-05-25 a 2020-06-23 (720 filas)
Bloque 19 OK: 2020-06-24 a 2020-07-23 (720 filas)
Bloque 20 OK: 2020-07-24 a 2020-08-22 (720 filas)
Bloque 21

In [ ]:
# Celda — Confirmar que el archivo quedo guardado en disco
import os
ruta = "../data/precio_bolsa_2019_2024.csv"
print("¿Existe el archivo?:", os.path.exists(ruta))
print("Tamaño:", os.path.getsize(ruta) / 1024, "KB")

¿Existe el archivo?: True
Tamaño: 1583.0517578125 KB


In [ ]:
# Celda — Buscar en el catalogo las metricas relacionadas con embalses/hidrologia

# Reutilizamos el objeto 'api' que ya creamos antes (api = ReadDB()).
# get_collections() devuelve el catalogo completo como DataFrame de pandas.
catalogo = api.get_collections()

# Buscamos en todas las columnas de texto cualquier fila que mencione "embalse".
# .apply recorre columna por columna; .astype(str).str.contains es metodo de pandas.
mascara_embalse = catalogo.apply(lambda columna: columna.astype(str).str.contains("embalse", case=False, na=False))

# Mostramos las columnas relevantes para identificar la metrica correcta.
print("=== Resultados para 'embalse' ===")
print(catalogo[mascara_embalse.any(axis=1)][["MetricId", "MetricName", "Entity", "Type", "MetricUnits"]])

# Tambien buscamos "aporte" y "hidrologia", porque XM suele separar estos conceptos
# en metricas distintas (volumen util, aportes hidricos, capacidad util, etc.)
mascara_aporte = catalogo.apply(lambda columna: columna.astype(str).str.contains("aporte", case=False, na=False))
print("\n=== Resultados para 'aporte' ===")
print(catalogo[mascara_aporte.any(axis=1)][["MetricId", "MetricName", "Entity", "Type", "MetricUnits"]])

NameError: name 'api' is not defined